# Phase 6: SOTA Push — Stage B
**Dataset**: emodb
**Model**: `dual_cnn_sa_lstm`
**Configuration**: 3 seeds (42, 123, 999) | 5 folds | Epochs: 15 | Label Smoothing: 0.05

### Goal
Statistically validate the Stage B baseline and self-attention models across 4 datasets for cross-lingual validation. 
*Note: This workload has been dynamically sharded by dataset and model to avoid the 12-hour Kaggle execution limit.*

In [ ]:
import os
import glob
import shutil
import zipfile
import subprocess
import sys
from pathlib import Path

WORK_DIR = Path('/kaggle/working/PROJECT')
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Find the dataset dynamically by looking for directories in /kaggle/input/
try:
    available_dirs = [d for d in os.listdir('/kaggle/input') if os.path.isdir(os.path.join('/kaggle/input', d))]
except FileNotFoundError:
    available_dirs = []

if not available_dirs:
    raise FileNotFoundError("NO DATASETS ATTACHED! Please click the + Add Data button in the right sidebar (Input tab).")

# Just grab the first dataset folder it finds (Kaggle mounts attached datasets here)
dataset_folder_name = available_dirs[0]
EXPECTED_DIR = Path('/kaggle/input') / dataset_folder_name

print(f"Dataset securely located at: {EXPECTED_DIR}")

# Look for a zip first, then assume auto-extracted
zips = list(EXPECTED_DIR.glob('*.zip'))

if zips:
    ZIP_PATH = zips[0]
    print(f'Found zip: {ZIP_PATH}')
    print('Extracting... (may take 2-3 mins)')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('/kaggle/working/')
    print('Done extracting.')
else:
    print(f'No zip found — scanning auto-extracted contents from {EXPECTED_DIR}...')
    # Kaggle sometimes nests the upload into another directory arbitrarily.
    # Find the actual 'product' directory containing our code & splits.
    product_dirs = list(EXPECTED_DIR.rglob('product'))
    
    if not product_dirs:
        raise FileNotFoundError(f"Could not find the 'product' directory inside {EXPECTED_DIR}!")
        
    actual_product_dir = product_dirs[0]
    
    # We want /kaggle/working/PROJECT/product/...
    target_product_dir = WORK_DIR / 'product'
    
    shutil.copytree(actual_product_dir, target_product_dir, dirs_exist_ok=True)
    print(f'Done copying from {actual_product_dir} to {target_product_dir}')

In [ ]:
os.chdir(WORK_DIR)
print(f'CWD: {os.getcwd()}')

ROOT = WORK_DIR
SPLITS = ROOT / 'product/artifacts/splits'

# ── Leakage guard: assert 0 subject overlap for every fold ──
import pandas as pd
leakage_found = False

for fold in range(5):
    train_csv = SPLITS / f'train_emodb_fold{fold}.csv'
    val_csv = SPLITS / f'val_emodb_fold{fold}.csv'
    if not train_csv.exists() or not val_csv.exists():
        print(f'WARN: fold{fold} CSVs missing, skipping check')
        continue
    
    # PC-GITA has patient_id, others have subject_id (usually)
    # Check column names dynamically
    t_df = pd.read_csv(train_csv)
    v_df = pd.read_csv(val_csv)
    subj_col = 'patient_id' if 'patient_id' in t_df.columns else ('subject_id' if 'subject_id' in t_df.columns else None)
    
    if not subj_col:
        print(f'WARN: no subject tracking column found for fold{fold}. Cannot verify leakage.')
        continue

    t_subj = set(t_df[subj_col])
    v_subj = set(v_df[subj_col])
    overlap = t_subj & v_subj
    if overlap:
        print(f'CRITICAL: fold{fold} has {len(overlap)} overlapping subjects: {overlap}')
        leakage_found = True
    else:
        print(f'fold{fold}: OK (train={len(t_subj)} subjects, val={len(v_subj)} subjects, overlap=0)')

if leakage_found:
    raise RuntimeError('DATA LEAKAGE DETECTED — aborting. Delete and re-upload the Kaggle dataset.')
print('All folds passed leakage check.')

In [ ]:
import shutil
runs_root = ROOT / "product/artifacts/runs"
cleaned = 0
PHASE6_TAGS = ["_resnet50_ca_lstm_", "_dual_cnn_sa_lstm_"]
for ds in ["emodb"]:
    ds_dir = runs_root / ds
    if not ds_dir.exists():
        continue
    for run_dir in list(ds_dir.iterdir()):
        name = run_dir.name
        if any(tag in f"_{name}_" for tag in PHASE6_TAGS):
            shutil.rmtree(run_dir)
            cleaned += 1
print(f"Cleaned {cleaned} stale Phase 6 run directories.")

In [ ]:
# ============================================================
# Phase 6 Stage B: emodb, 3 seeds, 5 folds
# Model: dual_cnn_sa_lstm (15 runs total)
# Label smoothing parameter: 0.05, Epochs: 15
# ============================================================
import sys
import subprocess
from pathlib import Path

ROOT = Path('/kaggle/working/PROJECT')
total = 15
done = 0
print(f"Phase 6 Stage B: {total} runs ( emodb | dual_cnn_sa_lstm | 3 seeds × 5 folds)")


run_name = "emodb_dual_cnn_sa_lstm_s42_fold0"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "0",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "emodb_dual_cnn_sa_lstm_s42_fold0",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s42_fold1"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "1",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "emodb_dual_cnn_sa_lstm_s42_fold1",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s42_fold2"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "2",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "emodb_dual_cnn_sa_lstm_s42_fold2",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s42_fold3"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "3",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "emodb_dual_cnn_sa_lstm_s42_fold3",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s42_fold4"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "4",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "42",
        "--run_name", "emodb_dual_cnn_sa_lstm_s42_fold4",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s123_fold0"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "0",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "123",
        "--run_name", "emodb_dual_cnn_sa_lstm_s123_fold0",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s123_fold1"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "1",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "123",
        "--run_name", "emodb_dual_cnn_sa_lstm_s123_fold1",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s123_fold2"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "2",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "123",
        "--run_name", "emodb_dual_cnn_sa_lstm_s123_fold2",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s123_fold3"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "3",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "123",
        "--run_name", "emodb_dual_cnn_sa_lstm_s123_fold3",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s123_fold4"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "4",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "123",
        "--run_name", "emodb_dual_cnn_sa_lstm_s123_fold4",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s999_fold0"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "0",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "999",
        "--run_name", "emodb_dual_cnn_sa_lstm_s999_fold0",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s999_fold1"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "1",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "999",
        "--run_name", "emodb_dual_cnn_sa_lstm_s999_fold1",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s999_fold2"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "2",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "999",
        "--run_name", "emodb_dual_cnn_sa_lstm_s999_fold2",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s999_fold3"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "3",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "999",
        "--run_name", "emodb_dual_cnn_sa_lstm_s999_fold3",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

run_name = "emodb_dual_cnn_sa_lstm_s999_fold4"
summary_path = ROOT / f"product/artifacts/runs/emodb/{run_name}/summary.json"

if summary_path.exists():
    print(f"SKIP {run_name} (already done)")
    done += 1
else:
    print(f"\n>>> RUNNING {run_name}")
    cmd = [
        sys.executable, "-u", "product/training/train_unified.py",
        "--dataset", "emodb",
        "--model_type", "dual_cnn_sa_lstm",
        "--fold", "4",
        "--epochs", "15",
        "--batch_size", "16",
        "--seed", "999",
        "--run_name", "emodb_dual_cnn_sa_lstm_s999_fold4",
        "--drop_last",
        "--lr", "0.0001",
        "--weight_decay", "0.01",
        "--spec_augment",
        "--label_smoothing", "0.05",
    ]

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()

    status = "DONE" if process.returncode == 0 else "ERROR"
    print(f"--- {status}: {run_name} ---")
    done += 1
    print(f"Progress: {done}/{total}")
    
    # [NEW] Incremental Zip - so we don't lose data if Kaggle times out!
    import zipfile
    output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
        runs_dir = ROOT / 'product/artifacts/runs'
        for f in runs_dir.glob('**/*'):
            if f.is_file() and f.suffix != '.pt':
                zipf.write(f, f.relative_to(runs_dir.parent.parent))
    print(f"Incremental zip updated: {output_zip}")

In [ ]:
# Aggregate Stage B results into a summary table
import json as _json
import numpy as np
from pathlib import Path as _P

print("\n=== Phase 6 Stage B Results (emodb | dual_cnn_sa_lstm) ===\n")
print(f"{'Model':<25} {'Fold':<6} {'Best F1':>8} {'Final F1':>9} {'AUC':>7}")
print("-" * 60)

runs_root = ROOT / "product/artifacts/runs/emodb"
model_type = "dual_cnn_sa_lstm"
SEEDS = [42, 123, 999]

for seed in SEEDS:
    fold_f1s = []
    print(f"SEED {seed}:")
    for fold in range(5):
        run_name = f"emodb_{model_type}_s{seed}_fold{fold}"
        summary_p = runs_root / run_name / "summary.json"
        if summary_p.exists():
            s = _json.loads(summary_p.read_text())
            f1  = s.get("best_macro_f1", 0.0)
            ff1 = s.get("final_macro_f1", 0.0)
            auc = s.get("final_auc", 0.0)
            fold_f1s.append(f1)
            print(f"{model_type:<25} {fold:<6} {f1:>8.4f} {ff1:>9.4f} {auc:>7.4f}")
        else:
            print(f"{model_type:<25} {fold:<6} {'MISSING':>8}")
    if fold_f1s:
        print(f"  → mean ± std: {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")
    print()

In [ ]:
# Zip all results for download
output_zip = f'/kaggle/working/phase6_emodb_dual_cnn_sa_lstm_results.zip'
print(f'Zipping results...')

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    runs_dir = ROOT / 'product/artifacts/runs'
    for f in runs_dir.glob('**/*'):
        if f.is_file() and f.suffix != '.pt':  # skip large model weights
            zipf.write(f, f.relative_to(runs_dir.parent.parent))

print(f'Done! Download {output_zip} from the Output tab.')
print(f'Stage B complete: 15 runs for emodb / dual_cnn_sa_lstm.')